# Building a Research Assistant with AutoGen

This notebook demonstrates a **multi-agent research assistant** using Microsoft AutoGen. You will build a team of agents that collaborate to:
- **Search** arXiv for papers on LLM applications in human learning
- **Plan** the workflow (Planner), **write code** (Engineer), **review content** (Scientist), and **critique** the plan (Critic)
- **Produce** a markdown table of papers by domain

## Learning objectives
- Configure LLM settings and load models from a config file
- Define specialized agents (Planner, Engineer, Scientist, Executor, Critic) and a human Admin
- Set up a GroupChat and GroupChatManager to orchestrate multi-agent conversations
- Run a research task end-to-end with the human approving the plan

In [ ]:
# ---------------------------------------------------------------------------
# Step 1: Install AutoGen (run once per environment)
# ---------------------------------------------------------------------------
# Uncomment the line below if AutoGen is not already installed.
# Use the version matching your course/material requirements.
# pip install autogen-agentchat==0.2.38

In [ ]:
# ---------------------------------------------------------------------------
# Step 2: Import AutoGen and verify version
# ---------------------------------------------------------------------------
# Verifying the version helps ensure compatibility with tutorial instructions.
import autogen
print(f"AutoGen version: {autogen.__version__}")

In [ ]:
# Version check merged into the previous cell — no need to run this cell.
# autogen.__version__

In [ ]:
# ---------------------------------------------------------------------------
# Step 3: Load LLM configuration
# ---------------------------------------------------------------------------
# Load model configuration from OAI_CONFIG_LIST.json (must be in the same directory).
# This file contains your API key and model name - keep it out of version control!
# Using the modern LLMConfig API instead of the deprecated config_list_from_json.
from pathlib import Path

config_file = "CONFIG_LIST.json"
if not Path(config_file).exists():
    raise FileNotFoundError(
        f"Config file '{config_file}' not found. "
        f"Create it in the same directory as this notebook. "
        f"See the markdown cell below for the expected format."
    )

# Load config using the modern LLMConfig API
llm_config_obj = autogen.LLMConfig.from_json(path=config_file)
config_list_gpt4 = llm_config_obj.config_list

# Validate that we got a valid config
if not config_list_gpt4:
    raise ValueError(
        f"Config file '{config_file}' is empty or invalid. "
        f"Ensure it contains at least one model configuration."
    )

# LLM settings for all agents:
# - temperature=0: deterministic outputs (good for reproducible results)
# - cache_seed: enables caching for faster/cheaper repeated runs
# - timeout: maximum seconds to wait for API responses
gpt4_config = {
    "cache_seed": 42,
    "temperature": 0.0,
    "config_list": config_list_gpt4,
    "timeout": 120,
}

### LLM configuration file format

The config file (`OAI_CONFIG_LIST.json`) must be in the same directory as this notebook. It lists one or more LLM endpoints with their API keys.

**📚 Reference:** [AutoGen LLM configuration documentation](https://microsoft.github.io/autogen/0.2/docs/topics/llm_configuration/)

**Example `OAI_CONFIG_LIST.json` format** (replace with your actual API key and model):

```json
[
  {
    "model": "gpt-4o-mini",
    "api_key": "sk-your-api-key-here"
  }
]
```

**⚠️ Security note:** Never commit this file to version control! Add `OAI_CONFIG_LIST.json` to your `.gitignore` file.

---

## Step 4: Define the multi-agent team

This research assistant uses **six specialized agents** that collaborate to complete the task:

1. **Admin** (UserProxyAgent): Human in the loop — approves plans and provides feedback
2. **Planner** (AssistantAgent): Creates and revises the workflow plan
3. **Engineer** (AssistantAgent): Writes Python code to scrape arXiv and format data
4. **Scientist** (AssistantAgent): Reviews papers and categorizes them by domain
5. **Executor** (UserProxyAgent): Runs code written by the Engineer
6. **Critic** (AssistantAgent): Reviews plans and outputs for quality and completeness

Each agent has a specific role, enabling the group to plan, implement, review, and critique the research task collaboratively.

In [ ]:
# Import the agent types and group chat components needed for multi-agent collaboration
from autogen import UserProxyAgent, AssistantAgent, GroupChat, GroupChatManager

# UserProxyAgent: Represents a human user or code executor (can run code)
# AssistantAgent: LLM-powered agent that generates responses and can write code
# GroupChat: Container that manages multiple agents and conversation flow
# GroupChatManager: Decides which agent speaks next based on the conversation context

In [ ]:
# Agent 1: Admin (Human Proxy)
# Represents the human user who approves plans and provides feedback.
# code_execution_config=False: This agent doesn't execute code, only approves plans.
user_proxy = UserProxyAgent(
    name="Admin",
    system_message="A human admin. Interact with the planner to discuss\
        the plan. Plan execution needs to be approved by this admin.",
    code_execution_config=False
)

In [ ]:
# Agent 2: Engineer (Code Writer)
# Writes Python/shell code to scrape arXiv, process data, and create markdown tables.
# Automatically fixes errors based on Executor feedback and iterates until code works.
engineer = AssistantAgent(
    name="Engineer",
    llm_config=gpt4_config,
    system_message="""Engineer. You follow an approved plan. You write python/shell code to solve tasks. Wrap the code in a code block that specifies the script type. The user can't modify your code. So do not suggest incomplete code which requires others to modify. Don't use a code block if it's not intended to be executed by the executor.
Don't include multiple code blocks in one response. Do not ask others to copy and paste the result. Check the execution result returned by the executor.
If the result indicates there is an error, fix the error and output the code again. Suggest the full code instead of partial code or code changes. If the error can't be fixed or if the task is not solved even after the code is executed successfully, analyze the problem, revisit your assumption, collect additional info you need, and think of a different approach to try.
""",
)

In [ ]:
# Agent 3: Scientist (Content Reviewer)
# Reviews paper abstracts and categorizes them by domain (e.g., Education, Healthcare).
# Does not write code — focuses on content analysis and categorization.
scientist = AssistantAgent(
    name="Scientist",
    llm_config=gpt4_config,
    system_message="""Scientist. You follow an approved plan.
    You are able to categorize papers after seeing their abstracts printed.
    You don't write code.""",
)

In [ ]:
# Agent 4: Planner (Workflow Designer)
# Creates the initial plan and revises it based on Admin and Critic feedback.
# Clearly assigns tasks to Engineer (code) vs Scientist (review) until Admin approves.
planner = AssistantAgent(
    name="Planner",
    system_message="""Planner. Suggest a plan. Revise the plan based on feedback
    from admin and critic, until admin approval. The plan may involve an engineer
    who can write code and a scientist who doesn't write code. Explain the plan 
    first. Be clear which step is performed by an engineer, and which step is 
    performed by a scientist.
    """,
    llm_config=gpt4_config
)

In [ ]:
# Agent 5: Executor (Code Runner)
# Automatically executes code written by the Engineer and reports results back.
# human_input_mode="NEVER": runs without prompting for user input
# work_dir="paper": creates a "paper" folder for all generated files and outputs
executor = UserProxyAgent(
    name="Executor",
    system_message="Executor. Execute the code written by the engineer\
        and report the result.",
    human_input_mode="NEVER",
    code_execution_config={"last_n_messages": 3,
                           "work_dir": "paper",
                           "use_docker" : False},
)

In [ ]:
# Agent 6: Critic (Quality Assurance)
# Reviews plans, code, and outputs from other agents to ensure quality.
# Ensures verifiable information (e.g., source URLs) is included in outputs.
critic = AssistantAgent(
    name="Critic", 
    system_message="""Critic. Double check plan, claims, code from other
    agents and provide feedback. Check whether the plan includes adding 
    verifiable info such as source URL.
    """,
    llm_config=gpt4_config,
)

In [ ]:
# ---------------------------------------------------------------------------
# Step 5: Create GroupChat and Manager
# ---------------------------------------------------------------------------
# GroupChat: Container that holds all agents and manages the conversation flow.
# max_round=10: Limits conversation to 10 rounds to prevent infinite loops.
groupchat = GroupChat(
    agents=[user_proxy, engineer, scientist, planner, executor, critic], 
    messages=[], 
    max_round=10
)

# GroupChatManager: Uses an LLM to decide which agent should speak next
# based on the conversation context and each agent's role.
manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

## Step 6: Run the research task

**How it works:**
1. The **Admin** sends the research request to the **manager**
2. The **manager** routes messages to the appropriate agents (Planner, Critic, Engineer, Scientist, Executor)
3. The **Planner** creates a plan, the **Critic** reviews it, and the **Admin** approves it
4. The **Engineer** writes code, the **Executor** runs it, and the **Scientist** reviews the results
5. The process continues until the task is complete or max_round is reached

**💡 Tip:** When prompted for plan approval, you can press **Enter** to auto-approve and let the agents continue automatically.

In [ ]:
# This cell is intentionally commented out — the manager is created in the cell above.
# Uncomment only if you need to recreate the manager after modifying the groupchat.
# manager = GroupChatManager(groupchat=groupchat, llm_config=gpt4_config)

In [ ]:
# Initiate the multi-agent conversation
# The Admin (user_proxy) sends the research task to the manager, which coordinates all agents.
user_proxy.initiate_chat(
    manager,
    message="""
    Find papers on LLM applications that can enhance or augment human learning from arxiv in the last week,
    create a markdown table of different domains.
    """,
)

# Note: The conversation will proceed automatically. You may be prompted to approve the plan;
# press Enter to auto-approve and continue, or type feedback to guide the agents.

---

## Expected output

Below is an **example** of the kind of markdown table the agents aim to produce. When you run the chat above to completion, the Engineer and Scientist will create a similar table from live arXiv results.

**Note:** This is a sample/placeholder table. The actual output will depend on:
- Papers available on arXiv matching your search criteria
- The current date (papers from "the last week")
- The agents' ability to categorize papers correctly

In [ ]:
# Example: Rendering a markdown table in the notebook
# This demonstrates how the final output table would look when rendered.
# The agents will produce similar output, but with real arXiv data.
from IPython.display import Markdown

# Sample markdown table (placeholder data)
text = """
| Title | Authors | Publication Date | Domain | URL |
|-------|---------|------------------|--------|-----|
| **Enhancing Student Learning with LLMs: A Case Study** | John Doe, Jane Smith | 2023-10-01 | Education | [arXiv URL](https://arxiv.org/abs/2310.00001) |
| **LLMs in Medical Diagnosis: Improving Accuracy and Efficiency** | Alice Johnson, Bob Lee | 2023-10-02 | Healthcare | [arXiv URL](https://arxiv.org/abs/2310.00002) |
| **Leveraging LLMs for Business Intelligence** | Carol White, David Brown | 2023-10-03 | Business | [arXiv URL](https://arxiv.org/abs/2310.00003) |
| **Social Impacts of LLMs in Online Communities** | Eve Black, Frank Green | 2023-10-04 | Social Sciences | [arXiv URL](https://arxiv.org/abs/2310.00004) |
| **Advancements in LLM Technology for Software Development** | Grace Blue, Henry Yellow | 2023-10-05 | Technology | [arXiv URL](https://arxiv.org/abs/2310.00005) |
| **Exploring LLMs in Creative Writing** | Ian Red, Julia Purple | 2023-10-06 | Others | [arXiv URL](https://arxiv.org/abs/2310.00006) |
"""

# Display the markdown table (renders nicely in Jupyter notebooks)
Markdown(text)

---

## ⚠️ Important disclaimer

**Experimental capabilities:** These multi-agent systems are still experimental. The example table above uses placeholder data.

**Hallucination risk:** In practice, LLM-generated content (including URLs, paper details, or categorizations) may not always match real sources. Always verify:
- URLs point to actual papers
- Paper titles and authors are accurate
- Domain categorizations are correct

**Best practices:**
- Review all agent-generated content before using it
- Cross-check URLs and references manually
- Validate that the research task was completed correctly
- Consider adding human review steps for production use